# NHS Patient Activity — ETL Pipeline


---

## What this notebook does

This notebook walks through a complete ETL pipeline for NHS patient activity data — from raw CSV extract through to a validated, idempotent load into a target database.

It is structured so that **every design decision is explained inline** alongside the code that implements it. The same pipeline logic is also unit-tested in `tests/` and executed automatically by the CI/CD workflow in `.github/workflows/ci.yml`.

### The five software practices demonstrated here

| Practice | Where |
|---|---|
| Separation of concerns (extract / transform / load) | `src/extract.py`, `src/transform.py`, `src/load.py` |
| Idempotent loads — safe to re-run any number of times | `src/load.py` · `tests/test_load.py` |
| Data quality validation with threshold alerting | `src/transform.py` · Cell 4 below |
| Unit tests covering all core logic | `tests/` directory |
| CI/CD — lint → test → execute notebook on every push | `.github/workflows/ci.yml` |


---
## 1. Environment setup

We import the three pipeline modules. Each module has a single responsibility and no knowledge of the others — this makes them independently testable and easy to swap out.


In [ ]:
import sys
import logging
from pathlib import Path

# Make sure the src modules are importable
sys.path.insert(0, str(Path.cwd().parent))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)

from src.extract import extract_csv
from src.transform import transform
from src.load import load, query_all

print("Modules loaded successfully")


---
## 2. Extract

We read the raw source file into a DataFrame. All columns are read as strings at this stage — no type casting happens in the extract layer.

**Design decision:** Keeping extraction thin means the source can be swapped (CSV → SQL Server → API) without touching the transform or load layers. The extract layer's only job is to get data into memory.


In [ ]:
raw_df = extract_csv("../data/mock_activity.csv")

print(f"Rows extracted: {len(raw_df)}")
print(f"Columns: {list(raw_df.columns)}")
raw_df


---
## 3. Transform

The transform layer applies all business logic and data quality rules. It returns two things:

1. A **clean DataFrame** ready for loading
2. A **ValidationReport** describing what was found

**Design decision:** Returning a report rather than raising exceptions gives the caller control. In production, the caller can decide whether to halt the pipeline, send an alert, or load the clean rows and quarantine the bad ones — without changing the transform logic itself.


In [ ]:
clean_df, report = transform(raw_df)

print(f"Rows in:  {report.total_rows}")
print(f"Rows out: {report.rows_passed}")
print()
print("── Data Quality Report ──────────────────────")
print(f"  Missing / invalid NHS numbers : {report.missing_nhs_numbers}  ({report.null_nhs_rate:.1%})")
print(f"  Open spells (no discharge)    : {report.missing_discharge_dates}")
print(f"  Duplicate admissions removed  : {report.duplicate_admissions}")
print()
if report.issues:
    print("Issues found:")
    for issue in report.issues:
        print(f"    {issue}")
else:
    print("  No issues found")


---
## 4. Threshold alerting — should we halt the pipeline?

A key pattern in NHS data pipelines is not just checking *whether* data quality issues exist, but checking *how bad* they are relative to a defined threshold.

**Design decision:** The threshold is configurable. In this example we gate on NHS number NULL rate. If more than 10% of rows are missing an NHS number, something is likely wrong upstream (feed failure, system change) and we halt rather than load bad data.

This is equivalent to setting a NULL rate alert in a CI/CD schema monitor.


In [4]:
NULL_THRESHOLD = 0.10  # 10% — configurable per environment

if report.exceeds_null_threshold(threshold=NULL_THRESHOLD):
    raise RuntimeError(
        f"Pipeline halted: NHS number NULL rate is {report.null_nhs_rate:.1%}, "
        f"which exceeds the {NULL_THRESHOLD:.0%} threshold. "
        f"Investigate the source feed before loading."
    )
else:
    print(f" NULL rate ({report.null_nhs_rate:.1%}) within threshold ({NULL_THRESHOLD:.0%}) — proceeding to load")


RuntimeError: Pipeline halted: NHS number NULL rate is 20.0%, which exceeds the 10% threshold. Investigate the source feed before loading.

---
## 5. Load — idempotent upsert

We write the clean rows to a SQLite database using `INSERT OR REPLACE`.

**Design decision: idempotency.**  
The pipeline can be re-run any number of times — the result is always identical. If a row already exists (matched on `patient_id + admission_date` primary key), it is replaced. If it does not exist, it is inserted. A plain `INSERT` would duplicate rows on every re-run.

In a production SQL Server environment this would use a `MERGE` statement. SQLite's `INSERT OR REPLACE` provides the same semantic guarantee.


In [ ]:
DB_PATH = "../data/pipeline_output.db"

rows_written = load(clean_df, DB_PATH)
print(f"Rows written: {rows_written}")


---
## 6. Verify idempotency — re-run the load

We run the load a second time with identical data to prove that re-running never creates duplicates. The row count in the target table must be the same after both runs.


In [ ]:
# Run the load again with the same data
load(clean_df, DB_PATH)

result_df = query_all(DB_PATH)
print(f"Row count after re-run: {len(result_df)}")
print()
assert len(result_df) == rows_written, (
    f"Idempotency violation: expected {rows_written} rows, found {len(result_df)}"
)
print("Idempotency confirmed — re-running produced identical output")


---
## 7. Inspect the target table


In [ ]:
result_df


---
## 8. Pipeline summary

| Step | Status |
|---|---|
| Extract |  Raw data loaded from CSV |
| Transform — date casting | Dates cast to datetime |
| Transform — NHS number validation | Missing / invalid NHS numbers flagged |
| Transform — open spell detection | Rows without discharge dates flagged |
| Transform — deduplication | Duplicate admissions removed |
| Threshold check | NULL rate within acceptable bounds |
| Load — idempotent upsert | Rows loaded safely |
| Idempotency verification | Re-run produced identical output |

---

## Running the tests

From the project root in a terminal:

```bash
pytest tests/ -v --cov=src --cov-report=term-missing
```

The CI/CD pipeline (`.github/workflows/ci.yml`) runs these automatically on every push and pull request — linting first, then the test suite, then a full notebook execution.
